# Training

Runs self-play training from this notebook and shows the network improving while it trains.

**How to use it**

1. Run the cells down to **Train**. The trainer starts a pool of worker processes and either builds a new
   network in `RUN_DIR` or resumes the one saved there.
2. The **Train** cell runs `GENERATIONS` generations. A dashboard above its output is redrawn after every
   generation. Press the notebook's interrupt button to stop at any time: the trainer shuts down cleanly, and
   the last finished generation is the one saved.
3. Run the **Train** cell again to carry on. Between runs you can change `trainer.config` (learning rate,
   simulations, buffer size, …); the next generation picks the change up, and each logged row records the
   settings it used.
4. The sections after **Train** go deeper: the full charts, how the network's view of the probe positions has
   changed, and individual self-play positions with the network's prior next to the search's policy target.
   Run them while training is stopped.

Each run directory holds `trained.pt` (latest network and optimizer), `checkpoints/genNNNN.pt` (snapshots),
`training_log.csv` (one row per generation) and `probes.jsonl`. To start from scratch, choose a new `RUN_DIR`.
The replay buffer is only kept in memory, so after a kernel restart training resumes with an empty buffer.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import matplotlib.pyplot as plt
import pandas as pd
from utt_smart.nn.training import plots
from utt_smart.nn.training.train import Trainer, TrainingConfig

plots.use_style()

## Settings

`TrainingConfig`'s defaults are: 400 self-play games per generation, a full
200-simulation search on a quarter of moves (the policy targets) and 40 simulations on the rest, an 8-generation
replay buffer, and 100 evaluation games against MCT(1000). A generation took about 2 minutes on this machine in
the last run.

Two settings matter for watching progress:

- **`mct_iterations`**: the plain MCT opponent. In the last run the network beat MCT(1000) more than 95% of the
  time from about generation 60 on, after which the win rate can't show further progress. Raise it when the
  win rate saturates (it is logged, so the chart's title shows which opponent each point used).
- **`snapshot_every`, `ladder_games`, `ladder_sims`**: every `snapshot_every` generations the network is saved
  and plays `ladder_games` games against the previous snapshot. The result gives an Elo rating, chained back to
  the untrained network at 0. Unlike the MCT win rate, it doesn't top out.

In [ ]:
RUN_DIR = "runs/from-scratch"   # a new directory starts a new network; an existing one resumes
WORKERS = None                  # worker processes for self-play and evaluation (None: one per core)

config = TrainingConfig(
    snapshot_every=5,   # snapshot and rate the network every 5 generations
    ladder_games=40,    # games against the previous snapshot (20 as X, 20 as O)
    ladder_sims=100,    # simulations per move for both networks in those games
)
config

In [ ]:
# Re-running this cell replaces the trainer and its worker pool
if "trainer" in globals():
    trainer.close()
trainer = Trainer(config, run_dir=RUN_DIR, workers=WORKERS)

## Train

The dashboard, after each generation:

- **Win rate vs MCT**: the headline number while the network is weaker than MCT. Dots are single generations;
  the line pools the last 5.
- **Elo**: rated at every snapshot by playing the previous one. Each step has an error of roughly ±60 Elo with
  40 games, and the steps add up, so read the trend.
- **Policy and value loss**: the blue training loss, the orange loss on the generation's new positions before
  training on them, and the dashed floor each loss can't go below. The gap to the floor is what is left to learn.
  Orange well above blue and drifting apart means the network is memorising the buffer.
- **Probe boards**: the network's raw policy (no search) on three fixed positions, with its value for the side to
  move. Watch the policy on *X wins with cell 8* concentrate on the outlined cell.

In [ ]:
GENERATIONS = 50

dashboard = plots.LiveDashboard(trainer)
trainer.run(GENERATIONS, on_generation=dashboard.update)

### Changing settings between runs

Change `trainer.config` and run **Train** again. For example:

In [ ]:
# trainer.config.lr = 3e-4              # lower the learning rate once the losses flatten
# trainer.config.mct_iterations = 5000  # a stronger MCT opponent once the win rate saturates
# trainer.config.buffer_generations = 12
trainer.config

## Charts in detail

Everything logged in this run directory, across restarts. A dashed vertical line marks where each run started,
since the settings may have changed there.

In [ ]:
log = plots.frame(trainer.history)
print(f"{len(log)} generations, {log.selfplay_games.sum():,} self-play games, "
      f"{log.generation_seconds.sum() / 3600:.1f} h of training")
log.groupby("run", sort=False).agg(
    first_generation=("generation", "min"), generations=("generation", "size"), lr=("lr", "last"),
    full_sims=("full_sims", "last"), mct_iterations=("mct_iterations", "last"),
    games_per_gen=("games_per_gen", "last"), buffer_generations=("buffer_generations", "last"))

### Win rate against plain MCT

Split by side: X moves first and wins more often, so a network that only wins as X has more to learn. With 100
evaluation games a single generation's dot can swing about 10 points either way.

In [ ]:
fig, ax = plt.subplots()
plots.plot_win_rate(ax, log, by_side=True)
plt.show()

### Elo ladder

Each point is a snapshot, rated from its match against the previous one. Both play their best move with
`ladder_sims` simulations and no noise, from a 2-move random opening that each pair of games shares with the
colours swapped. The table shows the matches behind the points.

In [ ]:
fig, ax = plt.subplots()
plots.plot_elo(ax, log)
plt.show()
log[log.elo.notna()][["generation", "ladder_opponent", "ladder_wins", "ladder_draws", "ladder_losses", "elo"]] \
    .astype({"ladder_opponent": "Int64", "ladder_wins": "Int64", "ladder_draws": "Int64", "ladder_losses": "Int64"}) \
    .round(0).set_index("generation")

### Training loss

- **Policy:** cross-entropy can't drop below the entropy of the MCTS visit-count targets, the dashed line. The
  gap between the two is what the network still has to learn; a flat gap means the policy head has stopped
  improving.
- **Value:** the dashed line is the loss from always predicting 0. A value loss near it means the network can't
  yet tell winning positions from losing ones.

Both floors move when the self-play data changes, so read the gap, not the raw loss.

In [ ]:
fig, (policy_ax, value_ax) = plt.subplots(1, 2, figsize=(11, 4))
plots.plot_loss(policy_ax, log, "policy")
plots.plot_loss(value_ax, log, "value")
plt.tight_layout()
plt.show()

### Self-play games

Who wins the games the network plays against itself, and how long they last. Results that lean heavily to one
side, or games that get much shorter, can mean the search has collapsed onto one line of play.

In [ ]:
fig, (result_ax, length_ax) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"wspace": 0.45})
plots.plot_selfplay(result_ax, length_ax, log)
plt.show()

### Time per generation

Training time grows until the replay buffer is full, because the number of batches scales with the buffer.
The ladder only runs on snapshot generations.

In [ ]:
fig, ax = plt.subplots()
plots.plot_time(ax, log)
plt.show()

### Latest generations

In [ ]:
recent = log.tail(10).assign(
    eval=lambda d: d.eval_wins.astype(str) + "-" + d.eval_draws.astype(str) + "-" + d.eval_losses.astype(str),
    minutes=lambda d: d.generation_seconds / 60,
)
recent[["generation", "eval", "win_rate_rolling", "elo", "policy_loss", "policy_gap", "value_loss",
        "value_target_mse", "game_length", "buffer_positions", "batches", "lr", "minutes"]].round(3) \
    .set_index("generation")

## What the network believes about the probe positions

The network's own policy and value on each probe position (no search), at evenly spaced generations from the
untrained network to now. These show learning that the losses average away: whether the network has learned
that X is favoured from the empty board, and whether it finishes a game when it can.

In [ ]:
for probe in trainer.probes:
    plots.probe_evolution(probe, trainer.probe_history)
    plt.show()

## Inside one self-play generation

`trainer.last_games` holds the newest generation's self-play games (empty until a generation has run in this
kernel). Each game is the record that training uses: the positions, the full searches' visit counts (the policy
targets) and the game result for the side to move (the value targets).

The first chart steps through one game. The grey square is the move just played; the shaded cells are the legal
moves.

In [ ]:
GAME = 0
record = trainer.last_games[GAME]
plots.show_game(record, positions=8)
plt.show()

The second puts the network's current policy (no search) next to what the full search on that move found. The
difference between the two is what the policy loss trains away. Positions that had only a fast search have no
policy target, so pick one from `full_search_moves`.

In [ ]:
full_search_moves = [i for i, full in enumerate(record.has_policy.tolist()) if full]
print("Moves with a full search:", [i + 1 for i in full_search_moves])

MOVE = full_search_moves[len(full_search_moves) // 2]   # an index into the game's positions, from 0
plots.show_position(trainer.bot, record, MOVE, trainer.device)
plt.show()

## Shut down

Stops the worker processes. The network and logs are already saved after every generation.

In [ ]:
trainer.close()